# Aula 04 — Roteadores: o LLM decide, o ToolNode executa

**Objetivo:** fechar a primeira metade do ciclo do agente. Ao final você deve
saber: o que `tools_condition` devolve, o que o `ToolNode` faz com cada pedido,
o que há dentro de um `ToolMessage` — e o que acontece quando a tool falha.

**Pré-requisitos:** Aula 02 (aresta condicional) e Aula 03 (`tool_calls` que
ninguém executava — agora alguém executa).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: o porteiro da lanchonete

O cliente fala. O **porteiro** (que entende português) decide para onde mandar:

- 'quanto deu meu pedido?' → manda para o **caixa** (que calcula de verdade)
- 'obrigado, tchau!' → manda para a **saída** (nada a calcular)

| Lanchonete | No LangGraph | O que faz de verdade |
|---|---|---|
| porteiro | **`tools_condition`** | lê o último `AIMessage`: tem `tool_calls`? devolve `'tools'`; senão, `'__end__'` |
| caixa | **`ToolNode([tools])`** | executa cada pedido em Python e anexa um `ToolMessage` por pedido |
| recibo grampeado no pedido | **`ToolMessage`** | `content` = resultado como string, `name` = tool usada, `tool_call_id` = id do pedido |

Tradução da Aula 02: `tools_condition` **é** uma aresta condicional comum —
só que a função roteadora já vem pronta, em vez de você escrever `decide_mood`.

In [ ]:
# CÉLULA 2 — o porteiro puro (sem LLM, 100% offline)
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import tools_condition
from langgraph.graph import END

com_pedido = {'messages': [HumanMessage(content='Quanto é 3x4?'),
    AIMessage(content='', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4},
                                       'id': 'c1', 'type': 'tool_call'}])]}
sem_pedido = {'messages': [HumanMessage(content='Oi'), AIMessage(content='Olá!')]}

print('com tool_calls ->', tools_condition(com_pedido))     # esperado: tools
print('sem tool_calls  ->', tools_condition(sem_pedido))    # esperado: __end__
assert tools_condition(com_pedido) == 'tools'
assert tools_condition(sem_pedido) == END
print('OK — o porteiro só lê tool_calls; ele nem sabe o que a tool faz.')

## 2. Conceito: as 3 peças e como se encaixam

```
START -> assistant --tools_condition--> tools -> END
              |                         (executa e anexa ToolMessage)
              +--- (sem tool_calls) ---> __end__
```

1. **`assistant`**: nó comum que invoca o LLM com cardápio (`bind_tools`).
   Devolve `AIMessage` com texto **ou** com `tool_calls`.
2. **`tools_condition`**: aresta condicional pronta. Não executa nada — só
   **lê** o último `AIMessage` e devolve a string de rota.
3. **`ToolNode([multiply, soma])`**: nó pronto que executa. Para **cada**
   `tool_call`, chama a função Python e anexa um `ToolMessage` com o resultado
   convertido para string (`12` vira `'12'`).

Repare no que **falta**: não há aresta de `tools` de volta para `assistant`.
Depois de executar, o grafo **encerra**. O LLM nunca vê o resultado — por isso
a resposta final desta aula é o `'12'` cru, sem frase. O ciclo completo
(mostrar o resultado ao LLM) é a Aula 05 (ReAct).

In [ ]:
# CÉLULA 3 — o grafo roteador completo (Groq real com chave, Fake sem)
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

MODELO = 'openai/gpt-oss-20b'

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

def soma(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

TOOLS = [multiply, soma]

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline.')
    return FakeMessagesListChatModel(responses=[AIMessage(
        content='',
        tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4},
                     'id': 'call_1', 'type': 'tool_call'}])])

def assistant(state: MessagesState) -> dict:
    return {'messages': [get_llm().invoke(state['messages'])]}

builder = StateGraph(MessagesState)
builder.add_node('assistant', assistant)
builder.add_node('tools', ToolNode(TOOLS))
builder.add_edge(START, 'assistant')
builder.add_conditional_edges('assistant', tools_condition)
builder.add_edge('tools', END)
grafo_roteador = builder.compile()

ultima = grafo_roteador.invoke(
    {'messages': [HumanMessage(content='Quanto é 3 vezes 4?')]})['messages'][-1]
print(type(ultima).__name__, repr(ultima.content))
assert ultima.content == '12', ultima.content
print('OK — o 12 finalmente foi CALCULADO (na Aula 03 ele nem existia).')

## 3. Dissecando o `ToolMessage` (o recibo)

- `content='12'` — **sempre string**, mesmo que a função devolva `int`.
  O `ToolNode` converte com `str()`. Na Aula 05 o LLM vai reler esse `'12'`.
- `name='multiply'` — qual tool rodou (o LLM confere se é a que ele pediu).
- `tool_call_id='fc_...'` — **igual ao `id` do pedido**. É o grampo que liga
  recibo a pedido. Se dois pedidos saem juntos, cada recibo carrega seu `id`.
- O `ToolMessage` entra no **histórico** (`messages`) como qualquer mensagem —
  por isso o `add_messages` da Aula 03 era pré-requisito: sem reducer, o recibo
  apagaria a conversa.

In [ ]:
# CÉLULA 4 — conversa simples: assistant -> END, sem tools
if os.getenv('GROQ_API_KEY'):
    out = grafo_roteador.invoke({'messages': [HumanMessage(content='Oi, tudo bem?')]})
    for m in out['messages']:
        print('[' + type(m).__name__ + ']', repr(m.content))
    assert [type(m).__name__ for m in out['messages']] == ['HumanMessage', 'AIMessage']
    print('OK — sem tool_calls, o porteiro mandou direto para o END.')
else:
    print('(no Fake offline o tool_call é sempre o mesmo gravado;')
    print(' com a chave Groq, esta célula mostra assistant -> END sem tools.)')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Pedir tool fora da lista **não quebra** — vira recibo de erro

Se o LLM pede `fantasma` e o `ToolNode` só conhece `[multiply, soma]`, o
resultado é um `ToolMessage` com texto de erro — e o grafo **encerra normal**.
Na Aula 05 isso importa: o LLM vai *ler* esse erro e pode se corrigir.

In [ ]:
# CÉLULA 5 — tool fora da lista (100% offline)
pedido_fantasma = AIMessage(content='', tool_calls=[
    {'name': 'fantasma', 'args': {}, 'id': 'c1', 'type': 'tool_call'}])

b = StateGraph(MessagesState)
b.add_node('tools', ToolNode(TOOLS))
b.add_edge(START, 'tools'); b.add_edge('tools', END)
out = b.compile().invoke({'messages': [pedido_fantasma]})
print(repr(out['messages'][-1].content))
assert 'not a valid tool' in out['messages'][-1].content
print('OK — erro virou mensagem, não exceção.')

### 4b. Exceção DENTRO da tool **quebra** a execução

O oposto do item anterior: se a função Python explode (`1/0`), o erro
**propaga** e o `invoke` morre com `ZeroDivisionError`. O `ToolNode` não
captura exceções da sua função. Lição de dev clássico: valide os `args`
antes de expor a tool — ou trate dentro dela e devolva string de erro.

In [ ]:
# CÉLULA 6 — tool que explode (100% offline)
def divide(a: int, b: int) -> float:
    """Divide a por b."""
    return a / b

pedido_zero = AIMessage(content='', tool_calls=[
    {'name': 'divide', 'args': {'a': 1, 'b': 0}, 'id': 'c2', 'type': 'tool_call'}])

b2 = StateGraph(MessagesState)
b2.add_node('tools', ToolNode([divide]))
b2.add_edge(START, 'tools'); b2.add_edge('tools', END)
try:
    b2.compile().invoke({'messages': [pedido_zero]})
except ZeroDivisionError as erro:
    print('ZeroDivisionError propagado:', erro)
    print('OK — exceção na tool não vira ToolMessage.')

### 4c. Dois pedidos de uma vez → dois recibos (um por `id`)

O `ToolNode` executa **todos** os `tool_calls` da resposta e anexa um
`ToolMessage` para cada — cada um grampeado no seu `tool_call_id`.

In [ ]:
# CÉLULA 7 — dois pedidos juntos (100% offline)
pedido_duplo = AIMessage(content='', tool_calls=[
    {'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'c3', 'type': 'tool_call'},
    {'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'c4', 'type': 'tool_call'}])

b3 = StateGraph(MessagesState)
b3.add_node('tools', ToolNode(TOOLS))
b3.add_edge(START, 'tools'); b3.add_edge('tools', END)
out = b3.compile().invoke({'messages': [pedido_duplo]})
for m in out['messages'][1:]:
    print(f'[ToolMessage id={m.tool_call_id}]', repr(m.content))
assert [(m.content, m.tool_call_id) for m in out['messages'][1:]] == [('6', 'c3'), ('20', 'c4')]
print('OK — um recibo por pedido, cada um com seu id.')

## 5. Erros clássicos

1. **Esperar a resposta final em frase.** Nesta aula o `END` chega logo após
   `tools` — o LLM nunca relê o `'12'`. Frase final com o resultado exige o
   ciclo de volta (Aula 05).
2. **Esquecer a tool no `ToolNode`.** `bind_tools([a, b])` com
   `ToolNode([a])`: pedir `b` vira recibo de erro (CÉLULA 5). Cardápio do LLM
   e lista do `ToolNode` devem andar juntos.
3. **Achar que `tools_condition` executa.** Ele só lê e roteia. Sem o nó
   `tools` registrado, a rota `'tools'` não leva a lugar nenhum.
4. **Não tratar exceção na tool.** Erro de Python propaga e mata o `invoke`
   (CÉLULA 6). Tools de produção validam args e devolvem erro como string.
5. **Confundir `id` do pedido com `name` da tool.** O grampo recibo-pedido é o
   `tool_call_id`; o `name` só diz qual função rodou.
6. **Ler `content` do `AIMessage` para decidir.** Com pedido, `content` vem
   vazio no `gpt-oss`. A decisão mora em `tool_calls` — é isso que o porteiro lê.

## 6. Exercício guiado — o caixa da lanchonete

Situação: o cliente pergunta o total de **3 pastéis a R$ 8** e depois dá
'obrigado, tchau!'. Mostre os dois trajetos completos (tipos de mensagem)
e confira: no primeiro o último é `ToolMessage` com `'24'`; no segundo,
o último é `AIMessage` sem `tool_calls`.

In [ ]:
# CÉLULA 8 — exercício guiado (rode e confira os trajetos)
def trajeto(pergunta: str):
    out = grafo_roteador.invoke({'messages': [HumanMessage(content=pergunta)]})
    tipos = [type(m).__name__ for m in out['messages']]
    print(pergunta, '->', tipos, '| último:', repr(out['messages'][-1].content))
    return out

out_conta = trajeto('Quanto é 3 vezes 8?')
if os.getenv('GROQ_API_KEY'):
    assert out_conta['messages'][-1].content == '24', out_conta['messages'][-1].content
    print('OK — o caixa calculou 3 x 8.')
else:
    assert out_conta['messages'][-1].content == '12'  # Fake: sempre multiply(3,4)
    print('(no Fake offline o tool_call é sempre o mesmo gravado;')
    print(' com a chave Groq, esta célula valida 3 x 8 = 24.)')


## 7. Desafio — porteiro com 3 tools e 3 destinos

Amplie o cardápio para `[multiply, soma, dividir]` (crie `dividir(a, b)`),
registre as 3 no `ToolNode` e teste:

- 'Quanto é 20 dividido por 4?' → `ToolMessage` com `'5.0'`
- 'some 10 com 5' → `ToolMessage` com `'15'`
- 'Oi, tudo bem?' → `AIMessage` sem `tool_calls` (só com a chave Groq real;
  no Fake offline o tool_call é sempre o mesmo gravado)

**Pegadinha (pense antes):** se você adicionar `dividir` no `bind_tools` mas
esquecer no `ToolNode`, o que o grafo devolve? (Resposta: recibo de erro da
CÉLULA 5 — e é exatamente assim que você debuga cardápio dessincronizado.)

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que `tools_condition` lê e quais são os dois destinos possíveis?
- [ ] O que o `ToolNode` faz com cada `tool_call` — e o que ele anexa?
- [ ] O que há dentro de um `ToolMessage` (`content`, `name`, `tool_call_id`)?
- [ ] Tool fora da lista: exceção ou mensagem? E exceção dentro da tool?
- [ ] Por que o `'12'` desta aula sai 'cru', sem frase — e o que falta para virar resposta?
- [ ] Cardápio do LLM e lista do `ToolNode` dessincronizados: qual o sintoma?

**Próxima aula (05 — Agente ReAct):** a aresta de volta `tools -> assistant`:
o LLM relê o recibo, pode pedir mais tools e só então encerra. É o loop
`pensar → agir → observar` que move o seu exemplo do pastel/refri/conta.